# 07. Formulación MILP de la capa Iota

Este notebook estudia la última transformación de una ronda de Keccak:
la capa `iota`.

Después de aplicar:

$$
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi,
$$

la transformación `iota` incorpora una constante pública dependiente del
índice de ronda.

La operación modifica únicamente el lane ubicado en $(0,0)$:

$$
A_{r+1}[0,0]
=
C_r[0,0]
\oplus
RC[r],
$$

donde:

- $C_r$ es la salida de `chi` en la ronda $r$;
- $RC[r]$ es la constante de la ronda;
- $A_{r+1}$ es el siguiente estado de frontera.

Los demás lanes se copian directamente:

$$
A_{r+1}[x,y,k]
=
C_r[x,y,k],
\qquad
(x,y)\neq(0,0).
$$

El objetivo del notebook será:

1. revisar las constantes de ronda truncadas para $z=4$ y $z=8$;
2. validar la implementación de referencia de `iota`;
3. comprobar el tamaño de la formulación MILP;
4. verificar que `iota` no crea variables nuevas;
5. comprobar su idempotencia;
6. resolver una ronda completa con CBC;
7. comparar el resultado MILP con la implementación de referencia.

In [1]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO DEL NOTEBOOK
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """
    Busca hacia arriba el directorio raíz del proyecto.

    Se considera raíz el directorio que contiene la carpeta `src`.
    """
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con una carpeta `src`."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [2]:
# ============================================================
# IMPORTACIONES Y VALIDACIÓN DE LA API DE IOTA
# ============================================================

import inspect
import numpy as np

import keccak_milp.layers as layers

from keccak_milp.config import ExperimentConfig
from keccak_milp.model import KeccakMILPModel


layer_names = [
    name
    for name in dir(layers)
    if (
        "iota" in name.lower()
        or "round_constant" in name.lower()
    )
]

model_names = [
    name
    for name in dir(KeccakMILPModel)
    if "iota" in name.lower()
]


print("Elementos disponibles en keccak_milp.layers:")

for name in layer_names:
    print(" -", name)


print("\nMétodos disponibles en KeccakMILPModel:")

for name in model_names:
    print(" -", name)


required_layer_names = {
    "ROUND_CONSTANTS_64",
    "round_constant",
    "iota",
}

required_model_names = {
    "_add_iota_constraints",
    "add_iota_layer",
    "iota_output_variable",
    "iota_output_values",
}

assert required_layer_names.issubset(
    set(layer_names)
)

assert required_model_names.issubset(
    set(model_names)
)


print("\nLa API de Iota está disponible correctamente.")

Elementos disponibles en keccak_milp.layers:
 - ROUND_CONSTANTS_64
 - iota
 - round_constant

Métodos disponibles en KeccakMILPModel:
 - _add_iota_constraints
 - add_iota_layer
 - iota_output_values
 - iota_output_variable

La API de Iota está disponible correctamente.


In [3]:
# ============================================================
# FIRMAS DE LA API DE IOTA
# ============================================================

round_constant = layers.round_constant
iota = layers.iota


objects_to_inspect = {
    "round_constant": round_constant,
    "iota": iota,
    "KeccakMILPModel.add_iota_layer": (
        KeccakMILPModel.add_iota_layer
    ),
    "KeccakMILPModel.iota_output_variable": (
        KeccakMILPModel.iota_output_variable
    ),
    "KeccakMILPModel.iota_output_values": (
        KeccakMILPModel.iota_output_values
    ),
}


for name, obj in objects_to_inspect.items():
    print(
        f"{name}{inspect.signature(obj)}"
    )

round_constant(round_index: 'int', z: 'int') -> 'int'
iota(state: 'NDArray[np.integer]', round_index: 'int') -> 'NDArray[np.int64]'
KeccakMILPModel.add_iota_layer(self, round_index: 'int') -> 'None'
KeccakMILPModel.iota_output_variable(self, round_index: 'int', x: 'int', y: 'int', k: 'int') -> 'pulp.LpVariable'
KeccakMILPModel.iota_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'


## Formulación MILP de Iota

Sea $C_r[x,y,k]$ la salida de `chi` en la ronda $r$ y sea
$A_{r+1}[x,y,k]$ el siguiente estado de frontera.

Para una posición fuera del lane $(0,0)$ se agrega una igualdad directa:

$$
A_{r+1}[x,y,k]
=
C_r[x,y,k].
$$

Para el lane $(0,0)$, cada bit depende del bit correspondiente de la
constante de ronda:

$$
A_{r+1}[0,0,k]
=
C_r[0,0,k]
\oplus
RC[r,k].
$$

Como $RC[r,k]$ es una constante binaria, existen dos casos.

Si:

$$
RC[r,k]=0,
$$

entonces:

$$
A_{r+1}[0,0,k]
=
C_r[0,0,k].
$$

Si:

$$
RC[r,k]=1,
$$

entonces:

$$
A_{r+1}[0,0,k]
=
1-C_r[0,0,k].
$$

Por tanto, la capa `iota` no necesita variables auxiliares ni variables
de salida adicionales. La salida se almacena directamente en el estado de
frontera $A_{r+1}$.

Para una longitud de lane $z$, la capa agrega:

$$
N_{\mathrm{variables}}^{\iota}=0,
$$

y:

$$
N_{\mathrm{restricciones}}^{\iota}=25z.
$$

In [4]:
# ============================================================
# CONSTANTES DE RONDA PARA z = 4 Y z = 8
# ============================================================

rounds_to_display = range(6)


print(
    "Ronda | RC de 64 bits       | z=4 | z=8"
)

print("-" * 45)


for round_index in rounds_to_display:
    full_constant = (
        layers.ROUND_CONSTANTS_64[
            round_index
        ]
    )

    constant_z4 = round_constant(
        round_index=round_index,
        z=4,
    )

    constant_z8 = round_constant(
        round_index=round_index,
        z=8,
    )

    print(
        f"{round_index:>5} | "
        f"0x{full_constant:016X} | "
        f"0x{constant_z4:X}   | "
        f"0x{constant_z8:02X}"
    )

Ronda | RC de 64 bits       | z=4 | z=8
---------------------------------------------
    0 | 0x0000000000000001 | 0x1   | 0x01
    1 | 0x0000000000008082 | 0x2   | 0x82
    2 | 0x800000000000808A | 0xA   | 0x8A
    3 | 0x8000000080008000 | 0x0   | 0x00
    4 | 0x000000000000808B | 0xB   | 0x8B
    5 | 0x0000000080000001 | 0x1   | 0x01


In [5]:
# ============================================================
# REPRESENTACIÓN BIT A BIT DE UNA CONSTANTE
# ============================================================

sample_round = 1
z = 8

sample_constant = round_constant(
    round_index=sample_round,
    z=z,
)

constant_bits = [
    (sample_constant >> k) & 1
    for k in range(z)
]


print("Ronda:", sample_round)
print(
    "Constante truncada:",
    f"0x{sample_constant:02X}",
)

print(
    "Bits en orden k = 0, ..., z - 1:",
    constant_bits,
)

print(
    "Posiciones activas:",
    [
        k
        for k, bit in enumerate(constant_bits)
        if bit == 1
    ],
)

Ronda: 1
Constante truncada: 0x82
Bits en orden k = 0, ..., z - 1: [0, 1, 0, 0, 0, 0, 0, 1]
Posiciones activas: [1, 7]


In [6]:
# ============================================================
# IOTA DE REFERENCIA SOBRE EL ESTADO NULO
# ============================================================

z = 8
sample_round = 1

zero_state = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

iota_zero_output = iota(
    zero_state.copy(),
    round_index=sample_round,
)


active_output_positions = [
    (x, y, k)
    for x in range(5)
    for y in range(5)
    for k in range(z)
    if iota_zero_output[x, y, k] == 1
]


print(
    "Peso del estado inicial:",
    int(zero_state.sum()),
)

print(
    "Peso después de Iota:",
    int(iota_zero_output.sum()),
)

print(
    "Bits activos después de Iota:",
    active_output_positions,
)


assert active_output_positions == [
    (0, 0, 1),
    (0, 0, 7),
]

print(
    "\nIota aplica correctamente RC[1] "
    "sobre el lane (0, 0)."
)

Peso del estado inicial: 0
Peso después de Iota: 2
Bits activos después de Iota: [(0, 0, 1), (0, 0, 7)]

Iota aplica correctamente RC[1] sobre el lane (0, 0).


In [7]:
# ============================================================
# IOTA SOLO MODIFICA EL LANE (0, 0)
# ============================================================

rng = np.random.default_rng(2026)

random_state = rng.integers(
    low=0,
    high=2,
    size=(5, 5, z),
    dtype=np.int64,
)

random_iota_output = iota(
    random_state.copy(),
    round_index=2,
)


unchanged_lanes = []

for x in range(5):
    for y in range(5):
        if (x, y) == (0, 0):
            continue

        if np.array_equal(
            random_iota_output[x, y, :],
            random_state[x, y, :],
        ):
            unchanged_lanes.append(
                (x, y)
            )


assert len(unchanged_lanes) == 24


print(
    "Lanes distintos de (0,0) sin cambios:",
    len(unchanged_lanes),
)

print(
    "Lane de entrada (0,0):",
    random_state[0, 0, :].tolist(),
)

print(
    "Lane de salida  (0,0):",
    random_iota_output[0, 0, :].tolist(),
)

Lanes distintos de (0,0) sin cambios: 24
Lane de entrada (0,0): [1, 0, 0, 1, 0, 0, 0, 0]
Lane de salida  (0,0): [1, 1, 0, 0, 0, 0, 0, 1]


## Validación estructural del modelo MILP

Para comprobar el costo estructural de `iota`, se construirá primero el
modelo hasta la salida de `chi`:

$$
A_r
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi.
$$

Después se agregará `iota` y se compararán los conteos.

Para $z=8$ se espera:

$$
25z=200
$$

restricciones nuevas y ninguna variable declarada adicional.

La salida de `iota` debe coincidir exactamente con:

$$
\texttt{state}[r+1,x,y,k].
$$

In [8]:
# ============================================================
# MODELO BASE: THETA → RHO-PI → CHI
# ============================================================

z = 8
round_index = 0

config = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

model_until_chi = KeccakMILPModel(
    config
)

model_until_chi.add_theta_layer(
    round_index
)

model_until_chi.add_rho_pi_layers(
    round_index
)

model_until_chi.add_chi_layer(
    round_index
)


declared_before_iota = (
    model_until_chi.declared_variable_count()
)

attached_before_iota = (
    model_until_chi.attached_variable_count()
)

constraints_before_iota = (
    model_until_chi.constraint_count()
)


print("Modelo construido hasta Chi")
print("-" * 45)

print(
    "Variables declaradas:",
    declared_before_iota,
)

print(
    "Variables conectadas:",
    attached_before_iota,
)

print(
    "Restricciones:",
    constraints_before_iota,
)

Modelo construido hasta Chi
---------------------------------------------
Variables declaradas: 1760
Variables conectadas: 1560
Restricciones: 1280


In [9]:
# ============================================================
# INCORPORACIÓN DE LA CAPA IOTA
# ============================================================

model_until_chi.add_iota_layer(
    round_index
)


declared_after_iota = (
    model_until_chi.declared_variable_count()
)

attached_after_iota = (
    model_until_chi.attached_variable_count()
)

constraints_after_iota = (
    model_until_chi.constraint_count()
)


added_declared_variables = (
    declared_after_iota
    - declared_before_iota
)

added_attached_variables = (
    attached_after_iota
    - attached_before_iota
)

added_constraints = (
    constraints_after_iota
    - constraints_before_iota
)


print("Incremento producido por Iota")
print("-" * 45)

print(
    "Variables declaradas añadidas:",
    added_declared_variables,
)

print(
    "Variables conectadas añadidas:",
    added_attached_variables,
)

print(
    "Restricciones añadidas:",
    added_constraints,
)

print(
    "Restricciones esperadas:",
    25 * z,
)


assert added_declared_variables == 0

assert added_constraints == 25 * z


print(
    "\nIota tiene el tamaño estructural esperado."
)

Incremento producido por Iota
---------------------------------------------
Variables declaradas añadidas: 0
Variables conectadas añadidas: 200
Restricciones añadidas: 200
Restricciones esperadas: 200

Iota tiene el tamaño estructural esperado.


In [10]:
# ============================================================
# VALIDACIÓN DE IDEMPOTENCIA
# ============================================================

declared_before_second_call = (
    model_until_chi.declared_variable_count()
)

attached_before_second_call = (
    model_until_chi.attached_variable_count()
)

constraints_before_second_call = (
    model_until_chi.constraint_count()
)


model_until_chi.add_iota_layer(
    round_index
)


declared_after_second_call = (
    model_until_chi.declared_variable_count()
)

attached_after_second_call = (
    model_until_chi.attached_variable_count()
)

constraints_after_second_call = (
    model_until_chi.constraint_count()
)


assert (
    declared_after_second_call
    == declared_before_second_call
)

assert (
    attached_after_second_call
    == attached_before_second_call
)

assert (
    constraints_after_second_call
    == constraints_before_second_call
)


print("Idempotencia verificada correctamente.")

print(
    "Variables declaradas antes y después:",
    declared_after_second_call,
)

print(
    "Variables conectadas antes y después:",
    attached_after_second_call,
)

print(
    "Restricciones antes y después:",
    constraints_after_second_call,
)

Idempotencia verificada correctamente.
Variables declaradas antes y después: 1760
Variables conectadas antes y después: 1760
Restricciones antes y después: 1480


In [11]:
# ============================================================
# SALIDA DE IOTA Y ESTADO DE FRONTERA SIGUIENTE
# ============================================================

sample_position = {
    "x": 2,
    "y": 3,
    "k": 4,
}


iota_variable = (
    model_until_chi.iota_output_variable(
        round_index=round_index,
        **sample_position,
    )
)

next_boundary_variable = (
    model_until_chi.state_variable(
        round_index=round_index + 1,
        **sample_position,
    )
)


assert iota_variable is next_boundary_variable


print(
    "Variable de salida Iota:",
    iota_variable.name,
)

print(
    "Variable del estado siguiente:",
    next_boundary_variable.name,
)

print(
    "¿Son el mismo objeto?:",
    iota_variable is next_boundary_variable,
)

print(
    "\nLa salida de Iota está conectada "
    "directamente con state[r + 1]."
)

Variable de salida Iota: a_r1_x2_y3_k4
Variable del estado siguiente: a_r1_x2_y3_k4
¿Son el mismo objeto?: True

La salida de Iota está conectada directamente con state[r + 1].


## Validación funcional de una ronda completa

La validación estructural confirmó que `iota` conecta la salida de `chi`
con el siguiente estado de frontera sin crear variables adicionales.

Ahora se comprobará el flujo completo:

$$
A_r
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi
\longrightarrow
\iota
\longrightarrow
A_{r+1}.
$$

Para una entrada binaria conocida se calcularán las referencias:

$$
T_{\mathrm{ref}}=\theta(A_r),
$$

$$
P_{\mathrm{ref}}=\rho\pi(T_{\mathrm{ref}}),
$$

$$
C_{\mathrm{ref}}=\chi(P_{\mathrm{ref}}),
$$

$$
A_{r+1,\mathrm{ref}}
=
\iota(C_{\mathrm{ref}},r).
$$

Después se resolverá el modelo con CBC y se exigirá igualdad bit a bit
en cada una de las etapas.

In [12]:
# ============================================================
# FUNCIONES AUXILIARES PARA LA VALIDACIÓN
# ============================================================

from pulp import LpStatus


def normalize_solution_state(state) -> np.ndarray:
    """
    Convierte una salida del modelo en un arreglo binario NumPy.
    """
    array = np.asarray(
        state,
        dtype=float,
    )

    if np.isnan(array).any():
        raise RuntimeError(
            "La solución contiene valores no definidos."
        )

    return np.rint(array).astype(
        np.int64
    )


def hamming_weight(state) -> int:
    """Calcula el peso de Hamming de un estado."""
    return int(
        np.asarray(
            state,
            dtype=np.int64,
        ).sum()
    )


def differing_positions(
    first_state,
    second_state,
) -> list[tuple[int, int, int, int, int]]:
    """
    Devuelve las posiciones diferentes entre dos estados.

    Cada elemento contiene:

        (x, y, k, primer_valor, segundo_valor)
    """
    first_array = np.asarray(
        first_state,
        dtype=np.int64,
    )

    second_array = np.asarray(
        second_state,
        dtype=np.int64,
    )

    if first_array.shape != second_array.shape:
        raise ValueError(
            "Los estados deben tener la misma forma."
        )

    differences = []

    for x in range(first_array.shape[0]):
        for y in range(first_array.shape[1]):
            for k in range(first_array.shape[2]):
                first_value = int(
                    first_array[x, y, k]
                )

                second_value = int(
                    second_array[x, y, k]
                )

                if first_value != second_value:
                    differences.append(
                        (
                            x,
                            y,
                            k,
                            first_value,
                            second_value,
                        )
                    )

    return differences


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


In [13]:
# ============================================================
# CONSTRUCCIÓN DE UNA ENTRADA CONTROLADA
# ============================================================

z = 8
round_index = 0

input_state = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

active_input_bits = [
    (0, 0, 0),
    (1, 0, 1),
    (0, 1, 3),
    (2, 3, 4),
    (3, 2, 5),
    (4, 4, 7),
]

for x, y, k in active_input_bits:
    input_state[x, y, k] = 1


assert input_state.shape == (5, 5, z)
assert np.all(np.isin(input_state, [0, 1]))
assert hamming_weight(input_state) > 0


print("Bits activos de la entrada:")

for position in active_input_bits:
    print(" -", position)

print(
    "\nPeso de Hamming de la entrada:",
    hamming_weight(input_state),
)

Bits activos de la entrada:
 - (0, 0, 0)
 - (1, 0, 1)
 - (0, 1, 3)
 - (2, 3, 4)
 - (3, 2, 5)
 - (4, 4, 7)

Peso de Hamming de la entrada: 6


In [14]:
# ============================================================
# RESULTADOS DE REFERENCIA DE UNA RONDA COMPLETA
# ============================================================

theta_reference = layers.theta(
    input_state.copy()
)

rho_pi_reference = layers.rho_pi(
    theta_reference.copy()
)

chi_reference = layers.chi(
    rho_pi_reference.copy()
)

iota_reference = layers.iota(
    chi_reference.copy(),
    round_index=round_index,
)


assert theta_reference.shape == (5, 5, z)
assert rho_pi_reference.shape == (5, 5, z)
assert chi_reference.shape == (5, 5, z)
assert iota_reference.shape == (5, 5, z)


print("Pesos de Hamming de referencia")
print("-" * 45)

print(
    "Entrada:",
    hamming_weight(input_state),
)

print(
    "Después de Theta:",
    hamming_weight(theta_reference),
)

print(
    "Después de Rho-Pi:",
    hamming_weight(rho_pi_reference),
)

print(
    "Después de Chi:",
    hamming_weight(chi_reference),
)

print(
    "Después de Iota:",
    hamming_weight(iota_reference),
)


assert (
    hamming_weight(theta_reference)
    ==
    hamming_weight(rho_pi_reference)
)

print(
    "\nRho-Pi conserva el peso de Hamming."
)

Pesos de Hamming de referencia
---------------------------------------------
Entrada: 6
Después de Theta: 66
Después de Rho-Pi: 66
Después de Chi: 89
Después de Iota: 88

Rho-Pi conserva el peso de Hamming.


In [15]:
# ============================================================
# DIFERENCIAS ENTRE CHI E IOTA
# ============================================================

chi_iota_differences = differing_positions(
    chi_reference,
    iota_reference,
)

active_constant_bits = [
    k
    for k in range(z)
    if (
        round_constant(
            round_index=round_index,
            z=z,
        )
        >> k
    ) & 1
]


print(
    "Bits activos de la constante:",
    active_constant_bits,
)

print(
    "Posiciones modificadas por Iota:"
)

for difference in chi_iota_differences:
    print(" -", difference)


assert len(chi_iota_differences) == len(
    active_constant_bits
)

assert all(
    x == 0 and y == 0
    for x, y, _, _, _ in chi_iota_differences
)

assert {
    k
    for _, _, k, _, _ in chi_iota_differences
} == set(active_constant_bits)


print(
    "\nIota solo modificó los bits indicados "
    "por la constante en el lane (0,0)."
)

Bits activos de la constante: [0]
Posiciones modificadas por Iota:
 - (0, 0, 0, 1, 0)

Iota solo modificó los bits indicados por la constante en el lane (0,0).


In [16]:
# ============================================================
# CONSTRUCCIÓN DEL MODELO DE UNA RONDA COMPLETA
# ============================================================

validation_config = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

validation_model = KeccakMILPModel(
    validation_config
)

validation_model.add_theta_layer(
    round_index
)

validation_model.add_rho_pi_layers(
    round_index
)

validation_model.add_chi_layer(
    round_index
)

validation_model.add_iota_layer(
    round_index
)


# ------------------------------------------------------------
# Fijar completamente el estado inicial A_0
# ------------------------------------------------------------

fixed_input_bits = 0

for x in range(5):
    for y in range(5):
        for k in range(z):
            input_variable = (
                validation_model.state_variable(
                    round_index=round_index,
                    x=x,
                    y=y,
                    k=k,
                )
            )

            validation_model.problem += (
                input_variable
                == int(input_state[x, y, k]),
                (
                    f"fix_round_input"
                    f"_r{round_index}"
                    f"_x{x}_y{y}_k{k}"
                ),
            )

            fixed_input_bits += 1


# Registrar la función objetivo requerida por solve().
validation_model.set_smoke_test_objective()


assert fixed_input_bits == 25 * z


print("Modelo de validación construido.")
print("-" * 45)

print(
    "Variables declaradas:",
    validation_model.declared_variable_count(),
)

print(
    "Variables conectadas:",
    validation_model.attached_variable_count(),
)

print(
    "Restricciones:",
    validation_model.constraint_count(),
)

print(
    "Bits de entrada fijados:",
    fixed_input_bits,
)

Modelo de validación construido.
---------------------------------------------
Variables declaradas: 1760
Variables conectadas: 1760
Restricciones: 1680
Bits de entrada fijados: 200


In [17]:
# ============================================================
# RESOLUCIÓN CON CBC
# ============================================================

solve_result = validation_model.solve()

status_code = validation_model.problem.status

status_name = LpStatus.get(
    status_code,
    str(status_code),
)


print(
    "Resultado devuelto por solve():",
    solve_result,
)

print(
    "Código de estado:",
    status_code,
)

print(
    "Estado del solver:",
    status_name,
)


assert status_name == "Optimal", (
    "Se esperaba una solución óptima, "
    f"pero CBC devolvió: {status_name}."
)

print(
    "\nLa ronda completa fue resuelta correctamente."
)

Resultado devuelto por solve(): Optimal
Código de estado: 1
Estado del solver: Optimal

La ronda completa fue resuelta correctamente.


In [18]:
# ============================================================
# RECUPERACIÓN DE LAS SALIDAS DEL MODELO
# ============================================================

theta_milp = normalize_solution_state(
    validation_model.theta_output_values(
        round_index
    )
)

rho_pi_milp = normalize_solution_state(
    validation_model.rho_pi_output_values(
        round_index
    )
)

chi_milp = normalize_solution_state(
    validation_model.chi_output_values(
        round_index
    )
)

iota_milp = normalize_solution_state(
    validation_model.iota_output_values(
        round_index
    )
)

next_boundary_state = np.asarray(
    [
        [
            [
                int(
                    round(
                        validation_model.state_variable(
                            round_index=round_index + 1,
                            x=x,
                            y=y,
                            k=k,
                        ).value()
                    )
                )
                for k in range(z)
            ]
            for y in range(5)
        ]
        for x in range(5)
    ],
    dtype=np.int64,
)


print("Pesos de Hamming obtenidos por el MILP")
print("-" * 45)

print(
    "Salida de Theta:",
    hamming_weight(theta_milp),
)

print(
    "Salida de Rho-Pi:",
    hamming_weight(rho_pi_milp),
)

print(
    "Salida de Chi:",
    hamming_weight(chi_milp),
)

print(
    "Salida de Iota:",
    hamming_weight(iota_milp),
)

print(
    "Estado de frontera A_1:",
    hamming_weight(next_boundary_state),
)

Pesos de Hamming obtenidos por el MILP
---------------------------------------------
Salida de Theta: 66
Salida de Rho-Pi: 66
Salida de Chi: 89
Salida de Iota: 88
Estado de frontera A_1: 88


In [19]:
# ============================================================
# COMPARACIÓN BIT A BIT DE TODA LA RONDA
# ============================================================

theta_differences = differing_positions(
    theta_milp,
    theta_reference,
)

rho_pi_differences = differing_positions(
    rho_pi_milp,
    rho_pi_reference,
)

chi_differences = differing_positions(
    chi_milp,
    chi_reference,
)

iota_differences = differing_positions(
    iota_milp,
    iota_reference,
)

boundary_differences = differing_positions(
    next_boundary_state,
    iota_reference,
)


print("Comparación bit a bit")
print("-" * 45)

print(
    "Diferencias en Theta:",
    len(theta_differences),
)

print(
    "Diferencias en Rho-Pi:",
    len(rho_pi_differences),
)

print(
    "Diferencias en Chi:",
    len(chi_differences),
)

print(
    "Diferencias en Iota:",
    len(iota_differences),
)

print(
    "Diferencias en A_1:",
    len(boundary_differences),
)


assert not theta_differences
assert not rho_pi_differences
assert not chi_differences
assert not iota_differences
assert not boundary_differences

assert np.array_equal(
    iota_milp,
    next_boundary_state,
)


print(
    "\nLa ronda MILP completa coincide "
    "con la implementación de referencia."
)

Comparación bit a bit
---------------------------------------------
Diferencias en Theta: 0
Diferencias en Rho-Pi: 0
Diferencias en Chi: 0
Diferencias en Iota: 0
Diferencias en A_1: 0

La ronda MILP completa coincide con la implementación de referencia.


## Validación con distintas constantes de ronda

La prueba anterior utilizó la ronda cero. Para comprobar que la formulación
selecciona correctamente diferentes constantes, se evaluarán casos con:

$$
r\in\{0,1,2\},
$$

y longitudes de lane:

$$
z\in\{4,8\}.
$$

No es necesario construir las rondas anteriores para esta prueba local.
El modelo ya declara todos los estados de frontera y puede fijarse
directamente $A_r$ antes de aplicar las capas correspondientes a la ronda
seleccionada.

In [20]:
# ============================================================
# FUNCIÓN REUTILIZABLE PARA UNA RONDA COMPLETA
# ============================================================

def solve_full_round_case(
    input_state: np.ndarray,
    round_index: int,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    str,
]:
    """
    Resuelve una ronda completa de Keccak reducido mediante MILP.

    Parameters
    ----------
    input_state:
        Estado binario con forma (5, 5, z).

    round_index:
        Índice de la ronda y de su constante Iota.

    Returns
    -------
    tuple
        Salidas de Theta, Rho-Pi, Chi, Iota y estado del solver.
    """
    input_array = np.asarray(
        input_state,
        dtype=np.int64,
    )

    if input_array.ndim != 3:
        raise ValueError(
            "El estado debe tener tres dimensiones."
        )

    if input_array.shape[:2] != (5, 5):
        raise ValueError(
            "Las dos primeras dimensiones deben ser 5 × 5."
        )

    local_z = input_array.shape[2]

    if local_z not in {4, 8}:
        raise ValueError(
            "El tamaño de palabra debe ser 4 u 8."
        )

    if round_index < 0:
        raise ValueError(
            "El índice de ronda no puede ser negativo."
        )

    if not np.all(
        np.isin(input_array, [0, 1])
    ):
        raise ValueError(
            "El estado debe ser binario."
        )

    local_config = ExperimentConfig(
        z=local_z,
        rounds=round_index + 1,
        solver="cbc",
        verbose=False,
    )

    local_model = KeccakMILPModel(
        local_config
    )

    local_model.add_theta_layer(
        round_index
    )

    local_model.add_rho_pi_layers(
        round_index
    )

    local_model.add_chi_layer(
        round_index
    )

    local_model.add_iota_layer(
        round_index
    )


    # --------------------------------------------------------
    # Fijar A_r
    # --------------------------------------------------------

    for x in range(5):
        for y in range(5):
            for k in range(local_z):
                variable = (
                    local_model.state_variable(
                        round_index=round_index,
                        x=x,
                        y=y,
                        k=k,
                    )
                )

                local_model.problem += (
                    variable
                    == int(input_array[x, y, k]),
                    (
                        f"fix_full_round_input"
                        f"_r{round_index}"
                        f"_x{x}_y{y}_k{k}"
                    ),
                )


    local_model.set_smoke_test_objective()

    status = local_model.solve()

    if status != "Optimal":
        raise RuntimeError(
            "CBC no encontró una solución óptima. "
            f"Estado obtenido: {status}."
        )


    theta_output = normalize_solution_state(
        local_model.theta_output_values(
            round_index
        )
    )

    rho_pi_output = normalize_solution_state(
        local_model.rho_pi_output_values(
            round_index
        )
    )

    chi_output = normalize_solution_state(
        local_model.chi_output_values(
            round_index
        )
    )

    iota_output = normalize_solution_state(
        local_model.iota_output_values(
            round_index
        )
    )


    return (
        theta_output,
        rho_pi_output,
        chi_output,
        iota_output,
        status,
    )


print(
    "Función reutilizable definida correctamente."
)

Función reutilizable definida correctamente.


In [21]:
# ============================================================
# VALIDACIÓN DE VARIAS RONDAS Y TAMAÑOS DE PALABRA
# ============================================================

validation_cases = [
    {
        "seed": 7,
        "z": 4,
        "round_index": 0,
    },
    {
        "seed": 2026,
        "z": 4,
        "round_index": 1,
    },
    {
        "seed": 640,
        "z": 8,
        "round_index": 0,
    },
    {
        "seed": 225,
        "z": 8,
        "round_index": 1,
    },
    {
        "seed": 701,
        "z": 8,
        "round_index": 2,
    },
]

validation_results = []


for case_number, case in enumerate(
    validation_cases,
    start=1,
):
    seed = case["seed"]
    local_z = case["z"]
    local_round = case["round_index"]

    rng = np.random.default_rng(seed)

    random_input = rng.integers(
        low=0,
        high=2,
        size=(5, 5, local_z),
        dtype=np.int64,
    )


    # --------------------------------------------------------
    # Referencia
    # --------------------------------------------------------

    theta_expected = layers.theta(
        random_input.copy()
    )

    rho_pi_expected = layers.rho_pi(
        theta_expected.copy()
    )

    chi_expected = layers.chi(
        rho_pi_expected.copy()
    )

    iota_expected = layers.iota(
        chi_expected.copy(),
        round_index=local_round,
    )


    # --------------------------------------------------------
    # MILP
    # --------------------------------------------------------

    (
        theta_obtained,
        rho_pi_obtained,
        chi_obtained,
        iota_obtained,
        solver_status,
    ) = solve_full_round_case(
        input_state=random_input,
        round_index=local_round,
    )


    theta_correct = np.array_equal(
        theta_obtained,
        theta_expected,
    )

    rho_pi_correct = np.array_equal(
        rho_pi_obtained,
        rho_pi_expected,
    )

    chi_correct = np.array_equal(
        chi_obtained,
        chi_expected,
    )

    iota_correct = np.array_equal(
        iota_obtained,
        iota_expected,
    )


    validation_results.append(
        {
            "caso": case_number,
            "semilla": seed,
            "z": local_z,
            "ronda": local_round,
            "constante": (
                f"0x{round_constant(local_round, local_z):X}"
            ),
            "solver": solver_status,
            "peso_entrada": hamming_weight(
                random_input
            ),
            "peso_theta": hamming_weight(
                theta_obtained
            ),
            "peso_rho_pi": hamming_weight(
                rho_pi_obtained
            ),
            "peso_chi": hamming_weight(
                chi_obtained
            ),
            "peso_iota": hamming_weight(
                iota_obtained
            ),
            "theta_correcto": theta_correct,
            "rho_pi_correcto": rho_pi_correct,
            "chi_correcto": chi_correct,
            "iota_correcto": iota_correct,
        }
    )


    assert solver_status == "Optimal"
    assert theta_correct
    assert rho_pi_correct
    assert chi_correct
    assert iota_correct


print(
    f"Se validaron correctamente "
    f"{len(validation_results)} casos."
)

Se validaron correctamente 5 casos.


In [22]:
# ============================================================
# RESUMEN DE LA VALIDACIÓN
# ============================================================

header = (
    "Caso | z | Ronda | RC   | Solver  | "
    "Entrada | Theta | Rho-Pi | Chi | Iota | Iota OK"
)

print(header)
print("-" * len(header))


for result in validation_results:
    print(
        f"{result['caso']:>4} | "
        f"{result['z']:>1} | "
        f"{result['ronda']:>5} | "
        f"{result['constante']:<4} | "
        f"{result['solver']:<7} | "
        f"{result['peso_entrada']:>7} | "
        f"{result['peso_theta']:>5} | "
        f"{result['peso_rho_pi']:>6} | "
        f"{result['peso_chi']:>3} | "
        f"{result['peso_iota']:>4} | "
        f"{str(result['iota_correcto']):>7}"
    )

Caso | z | Ronda | RC   | Solver  | Entrada | Theta | Rho-Pi | Chi | Iota | Iota OK
-----------------------------------------------------------------------------------
   1 | 4 |     0 | 0x1  | Optimal |      56 |    54 |     54 |  51 |   50 |    True
   2 | 4 |     1 | 0x2  | Optimal |      47 |    49 |     49 |  46 |   45 |    True
   3 | 8 |     0 | 0x1  | Optimal |      90 |    92 |     92 | 102 |  103 |    True
   4 | 8 |     1 | 0x82 | Optimal |      92 |   102 |    102 | 103 |  103 |    True
   5 | 8 |     2 | 0x8A | Optimal |     103 |   101 |    101 |  94 |   93 |    True


In [23]:
# ============================================================
# COMPROBACIÓN GLOBAL
# ============================================================

all_optimal = all(
    result["solver"] == "Optimal"
    for result in validation_results
)

all_theta_correct = all(
    result["theta_correcto"]
    for result in validation_results
)

all_rho_pi_correct = all(
    result["rho_pi_correcto"]
    for result in validation_results
)

all_chi_correct = all(
    result["chi_correcto"]
    for result in validation_results
)

all_iota_correct = all(
    result["iota_correcto"]
    for result in validation_results
)


assert all_optimal
assert all_theta_correct
assert all_rho_pi_correct
assert all_chi_correct
assert all_iota_correct


print("Todas las validaciones fueron superadas.")
print("-" * 45)

print(
    "Casos evaluados:",
    len(validation_results),
)

print(
    "Errores de Theta:",
    sum(
        not result["theta_correcto"]
        for result in validation_results
    ),
)

print(
    "Errores de Rho-Pi:",
    sum(
        not result["rho_pi_correcto"]
        for result in validation_results
    ),
)

print(
    "Errores de Chi:",
    sum(
        not result["chi_correcto"]
        for result in validation_results
    ),
)

print(
    "Errores de Iota:",
    sum(
        not result["iota_correcto"]
        for result in validation_results
    ),
)

Todas las validaciones fueron superadas.
---------------------------------------------
Casos evaluados: 5
Errores de Theta: 0
Errores de Rho-Pi: 0
Errores de Chi: 0
Errores de Iota: 0


## Conclusiones

La capa `iota` fue incorporada correctamente al modelo MILP de Keccak
reducido y permitió cerrar una ronda completa.

Los principales resultados son:

1. `iota` modifica únicamente el lane $(0,0)$ mediante:

   $$
   A_{r+1}[0,0]
   =
   C_r[0,0]
   \oplus
   RC[r].
   $$

2. Las constantes de 64 bits se truncan a los $z$ bits menos
   significativos para trabajar con $z=4$ y $z=8$.

3. Si el bit de la constante es cero, la formulación utiliza:

   $$
   A_{r+1}[0,0,k]
   =
   C_r[0,0,k].
   $$

4. Si el bit de la constante es uno, utiliza:

   $$
   A_{r+1}[0,0,k]
   =
   1-C_r[0,0,k].
   $$

5. La capa no crea nuevas variables. Reutiliza directamente las variables
   del siguiente estado de frontera:

   $$
   A_{r+1}.
   $$

6. Para una longitud de lane $z$, `iota` agrega exactamente:

   $$
   25z
   $$

   restricciones.

7. Para $z=8$, se agregan 200 restricciones y cero variables declaradas
   adicionales.

8. El método `add_iota_layer` es idempotente y requiere que `chi` haya
   sido agregada previamente.

9. La salida de `iota` coincide exactamente con el estado de frontera
   siguiente:

   $$
   \texttt{iota\_output}[r]
   =
   \texttt{state}[r+1].
   $$

10. CBC reprodujo bit a bit las implementaciones de referencia de
    `theta`, `rho_pi`, `chi` e `iota`.

11. La equivalencia se verificó para $z=4$, $z=8$ y diferentes índices
    de ronda.

Con esta etapa, el modelo representa una ronda completa:

$$
A_r
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi
\longrightarrow
\chi
\longrightarrow
\iota
\longrightarrow
A_{r+1}.
$$

El siguiente paso será construir automáticamente varias rondas consecutivas
y comprobar que la salida de una ronda se utiliza como entrada de la
siguiente.